In [0]:
# Configuration

CATALOG = "worldbank_ai"

BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

SOURCE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.indicator_metadata_raw"
)

TARGET_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.indicator_metadata"
)

print(f"Source: {SOURCE_TABLE}")
print(f"Target: {TARGET_TABLE}")

In [0]:
# Imports

from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
# Load Bronze indicator metadata

bronze_metadata_df = spark.table(
    SOURCE_TABLE
)

bronze_count = bronze_metadata_df.count()

print(
    f"Bronze indicator metadata records: "
    f"{bronze_count:,}"
)

bronze_metadata_df.printSchema()

display(
    bronze_metadata_df
)

In [0]:
# Inspect Bronze columns

print("Bronze indicator metadata columns:")
print("-" * 60)

for column_name in bronze_metadata_df.columns:
    print(column_name)

In [0]:
# Validate Bronze indicator metadata

null_indicator_codes = (
    bronze_metadata_df
    .filter(
        F.col("indicator_code").isNull()
        | (
            F.length(
                F.trim(
                    F.col("indicator_code")
                )
            ) == 0
        )
    )
    .count()
)

duplicate_indicator_codes_df = (
    bronze_metadata_df
    .groupBy(
        "indicator_code"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_indicator_codes = (
    duplicate_indicator_codes_df.count()
)

invalid_indicators = (
    bronze_metadata_df
    .filter(
        F.col("is_valid") != True
    )
    .count()
)

print(
    f"Null/empty indicator codes: "
    f"{null_indicator_codes}"
)

print(
    f"Duplicate indicator codes: "
    f"{duplicate_indicator_codes}"
)

print(
    f"Invalid indicators: "
    f"{invalid_indicators}"
)

if null_indicator_codes > 0:
    raise RuntimeError(
        "Null or empty indicator codes found."
    )

if duplicate_indicator_codes > 0:

    display(
        duplicate_indicator_codes_df
    )

    raise RuntimeError(
        "Duplicate indicator codes found."
    )

if invalid_indicators > 0:
    raise RuntimeError(
        "Invalid indicators found in Bronze metadata."
    )

print(
    "Bronze indicator metadata validation passed."
)

In [0]:
# Inspect configured and official indicator names

display(
    bronze_metadata_df
    .select(
        "indicator_code",
        "configured_name",
        "indicator_name",
        "unit"
    )
    .orderBy(
        "indicator_code"
    )
)

In [0]:
# Helper for Silver string normalization

def clean_string(column_name):

    return (
        F.when(
            F.col(column_name).isNull()
            | (
                F.length(
                    F.trim(
                        F.col(column_name)
                    )
                ) == 0
            ),
            None
        )
        .otherwise(
            F.trim(
                F.col(column_name)
            )
        )
    )

In [0]:
# Transform Bronze indicator metadata to Silver

silver_metadata_df = (
    bronze_metadata_df

    # -----------------------------------------------
    # Standardize identifiers
    # -----------------------------------------------

    .withColumn(
        "indicator_id",
        F.upper(
            F.trim(F.col("indicator_code"))
        )
    )

    # -----------------------------------------------
    # Clean names
    # -----------------------------------------------

    .withColumn(
        "display_name",
        clean_string("configured_name")
    )

    .withColumn(
        "official_name_clean",
        clean_string("indicator_name")
    )

    # -----------------------------------------------
    # Clean World Bank metadata
    # -----------------------------------------------

    .withColumn(
        "unit_clean",
        clean_string("unit")
    )

    .withColumn(
        "source_id_clean",
        clean_string("source_id")
    )

    .withColumn(
        "source_name_clean",
        clean_string("source_name")
    )

    .withColumn(
        "source_note_clean",
        clean_string("source_note")
    )

    .withColumn(
        "source_organization_clean",
        clean_string("source_organization")
    )

    # topic_names is ARRAY, not STRING.
    # Clean individual array elements instead of using trim()
    # on the whole array.
    .withColumn(
        "topic_names_clean",
        F.filter(
            F.transform(
                F.col("topic_names"),
                lambda x: F.trim(x)
            ),
            lambda x: x.isNotNull() & (F.length(x) > 0)
        )
    )

    # -----------------------------------------------
    # Silver processing metadata
    # -----------------------------------------------

    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )

    # -----------------------------------------------
    # Final Silver schema
    # -----------------------------------------------

    .select(
        "indicator_id",

        "display_name",

        F.col("official_name_clean").alias(
            "official_name"
        ),

        F.col("unit_clean").alias(
            "unit"
        ),

        F.col("source_id_clean").alias(
            "source_id"
        ),

        F.col("source_name_clean").alias(
            "source_name"
        ),

        F.col("source_note_clean").alias(
            "source_note"
        ),

        F.col("source_organization_clean").alias(
            "source_organization"
        ),

        # Original JSON representation
        "topics_json",

        # Clean array representation
        F.col("topic_names_clean").alias(
            "topic_names"
        ),

        "source_system",
        "source_endpoint",
        "ingested_at",
        "silver_processed_at"
    )
)

In [0]:
# Inspect Silver metadata

silver_metadata_df.printSchema()

display(
    silver_metadata_df
    .select(
        "indicator_id",
        "display_name",
        "official_name",
        "unit",
        "source_name",
        "topic_names"
    )
    .orderBy(
        "indicator_id"
    )
)

In [0]:
# Controlled indicator categories

indicator_category_map = {

    "NY.GDP.MKTP.KD.ZG":
        "GROWTH",

    "NY.GDP.PCAP.KD.ZG":
        "GROWTH",

    "NY.GDP.MKTP.CD":
        "ECONOMIC_SIZE",

    "NY.GDP.PCAP.CD":
        "INCOME_AND_OUTPUT",

    "FP.CPI.TOTL.ZG":
        "INFLATION",

    "NE.TRD.GNFS.ZS":
        "TRADE",

    "NE.EXP.GNFS.ZS":
        "TRADE",

    "NE.IMP.GNFS.ZS":
        "TRADE",

    "NE.GDI.TOTL.ZS":
        "INVESTMENT",

    "GC.XPN.TOTL.GD.ZS":
        "FISCAL",

    "GC.REV.XGRT.GD.ZS":
        "FISCAL",

    "GC.DOD.TOTL.GD.ZS":
        "FISCAL",

    "NY.GDS.TOTL.ZS":
        "SAVINGS",

    "BX.KLT.DINV.WD.GD.ZS":
        "EXTERNAL_SECTOR",

    "BN.CAB.XOKA.GD.ZS":
        "EXTERNAL_SECTOR"
}

In [0]:
category_expression = F.create_map(
    *[
        item
        for key, value
        in indicator_category_map.items()
        for item in (
            F.lit(key),
            F.lit(value)
        )
    ]
)

silver_metadata_df = (
    silver_metadata_df
    .withColumn(
        "indicator_category",
        category_expression[
            F.col("indicator_id")
        ]
    )
)

In [0]:
# Validate controlled category mapping

missing_category_df = (
    silver_metadata_df
    .filter(
        F.col(
            "indicator_category"
        ).isNull()
    )
)

missing_category_count = (
    missing_category_df.count()
)

print(
    f"Indicators without category: "
    f"{missing_category_count}"
)

if missing_category_count > 0:

    display(
        missing_category_df.select(
            "indicator_id",
            "display_name"
        )
    )

    raise RuntimeError(
        "One or more indicators are missing "
        "from the controlled category mapping."
    )

print(
    "Indicator category validation passed."
)

In [0]:
# Inspect indicator categories

display(
    silver_metadata_df
    .select(
        "indicator_id",
        "display_name",
        "indicator_category"
    )
    .orderBy(
        "indicator_category",
        "indicator_id"
    )
)

In [0]:
# Validate required Silver metadata

required_field_validation = (
    silver_metadata_df
    .select(

        F.sum(
            F.when(
                F.col(
                    "indicator_id"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_indicator_ids"
        ),

        F.sum(
            F.when(
                F.col(
                    "display_name"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_display_names"
        ),

        F.sum(
            F.when(
                F.col(
                    "official_name"
                ).isNull(),
                1
            ).otherwise(0)
        ).alias(
            "null_official_names"
        )
    )
    .collect()[0]
)

null_indicator_ids = (
    required_field_validation[
        "null_indicator_ids"
    ]
)

null_display_names = (
    required_field_validation[
        "null_display_names"
    ]
)

null_official_names = (
    required_field_validation[
        "null_official_names"
    ]
)

print(
    f"Null indicator IDs: "
    f"{null_indicator_ids}"
)

print(
    f"Null display names: "
    f"{null_display_names}"
)

print(
    f"Null official names: "
    f"{null_official_names}"
)

if (
    null_indicator_ids > 0
    or null_display_names > 0
    or null_official_names > 0
):

    raise RuntimeError(
        "Required Silver indicator metadata "
        "is missing."
    )

print(
    "Required metadata validation passed."
)

In [0]:
# Validate Silver indicator uniqueness

duplicate_indicators_df = (
    silver_metadata_df
    .groupBy(
        "indicator_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_count = (
    duplicate_indicators_df.count()
)

print(
    f"Duplicate Silver indicator IDs: "
    f"{duplicate_count}"
)

if duplicate_count > 0:

    display(
        duplicate_indicators_df
    )

    raise RuntimeError(
        "Duplicate Silver indicators detected."
    )

print(
    "Silver indicator uniqueness validation passed."
)

In [0]:
# Validate Bronze-to-Silver row count

silver_count = (
    silver_metadata_df.count()
)

print(
    f"Bronze records: {bronze_count:,}"
)

print(
    f"Silver records: {silver_count:,}"
)

if silver_count != bronze_count:

    raise RuntimeError(
        "Bronze-to-Silver indicator "
        "row count mismatch."
    )

if silver_count != 15:

    raise RuntimeError(
        f"Expected 15 governed indicators, "
        f"found {silver_count}."
    )

print(
    "Bronze-to-Silver row-count validation passed."
)

In [0]:
# Inspect optional metadata coverage

metadata_coverage_df = (
    silver_metadata_df
    .select(

        F.count("*").alias(
            "total_indicators"
        ),

        F.count(
            "unit"
        ).alias(
            "with_unit"
        ),

        F.count(
            "source_name"
        ).alias(
            "with_source_name"
        ),

        F.count(
            "source_note"
        ).alias(
            "with_source_note"
        ),

        F.count(
            "source_organization"
        ).alias(
            "with_source_organization"
        ),

        F.count(
            "topic_names"
        ).alias(
            "with_topics"
        )
    )
)

display(
    metadata_coverage_df
)

In [0]:
# Final pre-write validation

category_count = (
    silver_metadata_df
    .select(
        "indicator_category"
    )
    .distinct()
    .count()
)

print("=" * 60)
print("SILVER INDICATOR METADATA VALIDATION")
print("=" * 60)

print(
    f"Bronze records:       "
    f"{bronze_count:,}"
)

print(
    f"Silver records:       "
    f"{silver_count:,}"
)

print(
    f"Distinct indicators:  "
    f"{silver_count:,}"
)

print(
    f"Indicator categories: "
    f"{category_count:,}"
)

print(
    f"Missing categories:   "
    f"{missing_category_count:,}"
)

print(
    f"Duplicate IDs:        "
    f"{duplicate_count:,}"
)

print(
    f"Null indicator IDs:   "
    f"{null_indicator_ids:,}"
)

print(
    f"Null display names:   "
    f"{null_display_names:,}"
)

print(
    f"Null official names:  "
    f"{null_official_names:,}"
)

print(
    "\nAll Silver indicator metadata "
    "validations passed."
)

In [0]:
# Write Silver indicator metadata

(
    silver_metadata_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(
        TARGET_TABLE
    )
)

print(
    f"Saved Silver indicator metadata to:"
    f"\n{TARGET_TABLE}"
)

In [0]:
# Read-back validation

saved_silver_df = spark.table(
    TARGET_TABLE
)

saved_count = (
    saved_silver_df.count()
)

saved_distinct_indicators = (
    saved_silver_df
    .select(
        "indicator_id"
    )
    .distinct()
    .count()
)

print(
    f"Expected records: "
    f"{silver_count:,}"
)

print(
    f"Saved records:    "
    f"{saved_count:,}"
)

print(
    f"Distinct indicators: "
    f"{saved_distinct_indicators:,}"
)

if saved_count != silver_count:

    raise RuntimeError(
        "Silver indicator metadata "
        "write row-count validation failed."
    )

if saved_distinct_indicators != saved_count:

    raise RuntimeError(
        "Silver indicator uniqueness "
        "changed after write."
    )

print(
    "Silver indicator metadata write validated."
)

In [0]:
# Final summary

print("=" * 70)
print("WORLD BANK INDICATOR METADATA SILVER TRANSFORMATION")
print("=" * 70)

print(
    f"Source:               "
    f"{SOURCE_TABLE}"
)

print(
    f"Target:               "
    f"{TARGET_TABLE}"
)

print(
    f"Bronze records:       "
    f"{bronze_count:,}"
)

print(
    f"Silver records:       "
    f"{saved_count:,}"
)

print(
    f"Indicator categories: "
    f"{category_count:,}"
)

print(
    f"Duplicate IDs:        "
    f"{duplicate_count:,}"
)

print(
    f"Missing categories:   "
    f"{missing_category_count:,}"
)

print(
    "Layer:                Silver"
)

print(
    "Status:               SUCCESS"
)